# 36 — Evaluación final TEST y capacidad de revisión

**Este es el único notebook de M4 autorizado a consumir TEST.** La selección de modelo y calibrador ya debe estar congelada. La primera ejecución materializa la evaluación final; ejecuciones posteriores reutilizan los artefactos completos y no vuelven a evaluar TEST.

In [ ]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec
from argos.config.modeling import load_modeling_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m3_gate = paths.docs / "decisions" / "M3_GATE.md"
assert m3_gate.exists() and "M3 ✅ CLOSED" in m3_gate.read_text(encoding="utf-8"), (
    "M3 no está formalmente cerrado. No iniciar Modeling."
)
GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")
MODEL_SPEC = load_modeling_spec(paths.configs / "modeling.toml")
MODEL_DIR = paths.root / "artifacts" / "models" / "m4"
MANIFEST_DIR = paths.root / "artifacts" / "manifests"

{
    "gold_schema": GOLD_SPEC.schema_version,
    "modeling_schema": MODEL_SPEC.schema_version,
    "target": MODEL_SPEC.target,
    "model_matrix": MODEL_SPEC.model_matrix,
    "operational_review_fraction": MODEL_SPEC.ranking.operational_fraction,
}

assert (paths.reports / "modeling" / "35_validation.parquet").exists(), "Ejecuta 35 primero."
from argos.models.business import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION


In [ ]:
from argos.models.business import evaluate_final_test

result = evaluate_final_test(
    root=paths.root,
    gold_spec=GOLD_SPEC,
    modeling_spec=MODEL_SPEC,
    reports_root=paths.reports,
    model_dir=MODEL_DIR,
    manifest_dir=MANIFEST_DIR,
    docs_root=paths.docs,
)
{"reused_existing_test_artifacts": result.reused, "model_card": str(result.model_card_path)}


In [ ]:
result.metrics

In [ ]:
result.ranking

In [ ]:
result.reliability

In [ ]:
result.validation

In [ ]:
blocking = result.validation.loc[result.validation["blocking"]]
print("36 ✅ PASS" if result.ready_for_gate else "36 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Final TEST manifest:", result.manifest_path)
assert result.ready_for_gate
